## Agentic AI: Code Review Agent

**1. Aşama:** Basit LLM Call

**2. Aşama:** LLM Call + Tool

**3. Aşama:** LLM Call + Tool + More Agent (Workflow)

**SDK:** `google-genai` 

**Model:** `gemini-3-flash-preview`

### Kurulum

In [1]:
#!pip install -q google-genai python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3-flash-preview"

# Model parametreleri
# temperature      : yaratıcılık/tutarlılık dengesi (0.0 = deterministik, 2.0 = çok yaratıcı)
# top_p            : nucleus sampling — olasılık kütlesinin kaçta kaçı kapsansın (0.0-1.0)
# top_k            : her adımda kaç token aday gösterilsin
# max_output_tokens : yanıttaki maksimum token sayısı
# thinking_budget  : modelin düşünmek için kullanabileceği token bütçesi
#                    0 = düşünme kapalı, -1 = otomatik, 1-24576 = sabit bütçe
# include_thoughts : düşünce özetlerini yanıta dahil et
TEMPERATURE = 0.7
TOP_P = 0.95
TOP_K = 40
MAX_OUTPUT_TOKENS = 2048
THINKING_BUDGET = 1024
INCLUDE_THOUGHTS = False

### Aşama 1: Basit Agent

In [ ]:
REVIEWER_SYSTEM_PROMPT = """
Sen senior bir Python developersın.
Verilen kodu inceleyerek şunları sun:
1. Kodun ne yaptığı
2. Bulunan hatalar ve sorunlar
3. 1 ile 10 arasında bir puan 

Kısa ve yapılandırılmış olarak yanıtla. Türkçe yanıt ver.
"""

In [4]:
def simple_review(code: str) -> str:
    response = client.models.generate_content(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=REVIEWER_SYSTEM_PROMPT,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            thinking_config=types.ThinkingConfig(
                thinking_budget=THINKING_BUDGET,
                include_thoughts=INCLUDE_THOUGHTS,
            ),
        ),
        contents=f"Bu kodu incele:\n\n```python\n{code}\n```"
    )
    return response.text

In [5]:
sample_code = """
def find_max(lst):
    max_val = 0
    for x in lst:
        if x > max_val:
            max_val = x
    return max_val
"""

print(simple_review(sample_code))

Merhaba, bir Senior Python Developer olarak kodunu inceledim. İşte analizim:

### 1. Kodun Ne Yaptığı
Bu fonksiyon, kendisine verilen bir liste (`lst`) içerisindeki en büyük sayısal değeri bulmayı amaçlayan bir döngü mekanizmasıdır.

### 2. Bulunan Hatalar ve Sorunlar
*   **Negatif Sayı Hatası:** `max_val = 0` başlangıç değeri büyük bir mantık hatasıdır. Eğer liste sadece negatif sayılardan oluşuyorsa (örneğin: `[-5, -10, -2]`), fonksiyon yanlış bir şekilde `0` sonucunu döndürecektir. Doğru yaklaşım `float('-inf')` veya listenin ilk elemanını (`lst[0]`) başlangıç değeri seçmektir.
*   **Boş Liste Durumu:** Liste boş gönderilirse fonksiyon `0` döner. Bu durum, listenin içinde gerçekten `0` olduğu durumla karışabilir ve hatalı sonuçlara yol açabilir.
*   **Built-in Fonksiyon Kullanımı:** Python'da bu işlem için halihazırda optimize edilmiş `max()` fonksiyonu bulunmaktadır. Manuel döngü hem daha yavaş hem de daha fazla hata payı barındırır.
*   **Tip Belirtimi (Type Hinting):** Fonksiyonu

### Aşama 2: Tool Kullanan Agent


In [6]:
import subprocess
import sys


def run_python_code(code: str) -> str:
    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
        timeout=10
    )
    return result.stdout + result.stderr

In [7]:
code_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="run_python_code",
            description="Python kodunu çalıştırır ve stdout ile stderr çıktısını döner",
            parameters=types.Schema(
                type="OBJECT",
                properties={"code": types.Schema(type="STRING")},
                required=["code"]
            )
        )
    ]
)

In [8]:
def tool_review(code: str) -> str:
    messages = [f"Bu kodu incele ve test et:\n\n```python\n{code}\n```"]

    while True:
        response = client.models.generate_content(
            model=MODEL,
            config=types.GenerateContentConfig(
                system_instruction=REVIEWER_SYSTEM_PROMPT,
                tools=[code_tool],
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                thinking_config=types.ThinkingConfig(
                    thinking_budget=THINKING_BUDGET,
                    include_thoughts=INCLUDE_THOUGHTS,
                ),
            ),
            contents=messages
        )

        part = response.candidates[0].content.parts[0]

        if part.function_call:
            fc = part.function_call
            print(f"Action:      {fc.name}({dict(fc.args)})")

            result = run_python_code(**fc.args)
            print(f"Observation: {result.strip()[:300]}")
            print()

            messages.append(response.candidates[0].content)
            messages.append(types.Content(
                role="tool",
                parts=[types.Part(function_response=types.FunctionResponse(
                    name=fc.name,
                    response={"result": result}
                ))]
            ))
        else:
            return response.text

In [9]:
buggy_code = """
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n)  # n eksiltilmiyor

print(factorial(5))
"""

print(tool_review(buggy_code))

Senior Python Developer olarak incelemem aşağıdadır:

### 1. Kodun Ne Yaptığı
Bu kod, özyinelemeli (recursive) bir yöntemle bir sayının faktöriyelini hesaplamayı amaçlamaktadır. `n` değeri 0 olana kadar fonksiyonun kendisini çağırması hedeflenmiştir.

### 2. Bulunan Hatalar ve Sorunlar
*   **Sonsuz Özyineleme (Infinite Recursion):** En kritik hata, `factorial(n)` çağrısında `n` değerinin azaltılmamasıdır. Fonksiyon her seferinde aynı `n` değeriyle kendisini çağırır, bu da `n == 0` olan durdurma koşuluna (base case) asla ulaşılamayacağı anlamına gelir.
*   **RecursionError:** Kod çalıştırıldığında Python'un varsayılan özyineleme derinliği sınırı aşılacak ve `RecursionError: maximum recursion depth exceeded` hatası alınacaktır.
*   **Mantıksal Eksiklik:** Doğru çalışma için `factorial(n - 1)` şeklinde çağrılmalıdır.
*   **Girdi Kontrolü:** Fonksiyon negatif sayılar için bir kontrol içermemektedir; negatif bir sayı girilseydi (mantık doğru olsa bile) yine sonsuz döngüye girerdi.

**Düzelt


### Aşama 3: Multi-Agent

Tek agent tüm review'ı yapabilir, ama her şeyi eşit kalitede yapmaz. Görevi uzman agent'lara bölelim:

- **Correctness Agent** → bug, mantık hatası, güvenlik — kodu çalıştırarak test eder
- **Efficiency Agent** → performans, karmaşıklık, Pythonic stili — statik analiz yapar
- **Reporter** → iki raporu alır, sentezler ve final raporu üretir

Her agent farklı araç setine sahip olabilir. Bu, multi-agent mimarisinin en güçlü özelliklerinden biridir.



In [10]:
CORRECTNESS_PROMPT = """
Sen bir Python kodun doğru çalıştığından emin olan kişisin. Yalnızca aşağıdakilere odaklan:
- Mantık hataları ve böcekler (bug)
- Ele alınmamış uç durumlar (edge case)
- Güvenlik açıkları

Yanıtını şu JSON formatında ver: {"sorunlar": [...], "puan": 1-10}
Türkçe yaz.
"""

EFFICIENCY_PROMPT = """
Sen bir Python kodunun ne kadar clean/verimli yazıldığından sorumlu olan kişisin. Yalnızca aşağıdakilere odaklan:
- Zaman ve bellek karmaşıklığı
- Pythonic yazım kalıpları ve en iyi pratikler
- Gereksiz ya da tekrar eden işlemler

Yanıtını şu JSON formatında ver: {"sorunlar": [...], "puan": 1-10}
Türkçe yaz.
"""

REPORTER_PROMPT = """
Sen senior bir Python developersın.
İki uzman ajanın kod inceleme raporlarını alıyorsun.
Bu raporları birleştirerek aşağıdaki yapıda tek bir final rapor hazırla:
- Genel değerlendirme
- Tüm sorunlar (türüne göre gruplandırılmış)
- Genel puan (iki puanın ortalaması)
- En kritik 3 öneri
Türkçe yaz.
"""

In [11]:
def call_agent(system_prompt: str, message: str, tools: list = None) -> str:
    """
    tools verilirse ReAct döngüsüyle çalışır.
    Verilmezse tek seferlik çağrı yapar.
    """
    config = types.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=tools,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        thinking_config=types.ThinkingConfig(
            thinking_budget=THINKING_BUDGET,
            include_thoughts=INCLUDE_THOUGHTS,
        ),
    )

    messages = [message]

    while True:
        response = client.models.generate_content(
            model=MODEL,
            config=config,
            contents=messages
        )

        part = response.candidates[0].content.parts[0]

        if tools and part.function_call:
            fc = part.function_call
            print(f"  Action:      {fc.name}({dict(fc.args)})")
            result = run_python_code(**fc.args)
            print(f"  Observation: {result.strip()[:200]}")

            messages.append(response.candidates[0].content)
            messages.append(types.Content(
                role="tool",
                parts=[types.Part(function_response=types.FunctionResponse(
                    name=fc.name,
                    response={"result": result}
                ))]
            ))
        else:
            return response.text


def multi_agent_review(code: str) -> str:
    prompt = f"Bu kodu incele:\n\n```python\n{code}\n```"

    # Correctness Agent kodu çalıştırarak test eder
    print("[1/3] Correctness Agent...")
    correctness = call_agent(CORRECTNESS_PROMPT, prompt, tools=[code_tool])
    print(correctness)

    # Efficiency Agent statik analiz yapar, araç gerekmez
    print("[2/3] Efficiency Agent...")
    efficiency = call_agent(EFFICIENCY_PROMPT, prompt)
    print(efficiency)

    print("[3/3] Reporter...")
    final = call_agent(
        REPORTER_PROMPT,
        f"Doğruluk Raporu:\n{correctness}\n\nVerimlilik Raporu:\n{efficiency}"
    )
    return final

In [12]:
final_code = """
def find_duplicates(lst):
    duplicates = []
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j] and lst[i] not in duplicates:
                duplicates.append(lst[i])
    return duplicates
"""

report = multi_agent_review(final_code)
print(report)

[1/3] Correctness Agent...


```json
{
  "sorunlar": [
    "Zaman Karmaşıklığı: İç içe döngüler kullanıldığı için algoritma O(n^2) karmaşıklığına sahiptir. Büyük listelerde performans çok ciddi şekilde düşer.",
    "Gereksiz Arama: 'lst[i] not in duplicates' kontrolü, her eşleşmede 'duplicates' listesini baştan sona tarar. Bu da performansı negatif etkileyen ek bir O(k) maliyeti getirir.",
    "Verimlilik: Bir küme (set) veya sözlük (dict) kullanılarak O(n) zaman karmaşıklığı ile çok daha hızlı bir çözüm üretilebilir.",
    "Bellek ve DoS Riski: Çok büyük veri setleri gönderildiğinde işlemciyi aşırı yorarak sistemin yanıt vermemesine (Denial of Service) neden olabilir."
  ],
  "puan": 4
}
```
[2/3] Efficiency Agent...


```json
{
  "sorunlar": [
    "Zaman Karmaşıklığı: İç içe geçmiş döngüler (nested loops) nedeniyle algoritma O(N^2) zaman karmaşıklığına sahiptir. Liste boyutu büyüdükçe performans hızla düşer.",
    "Üyelik Kontrolü Verimsizliği: 'not in duplicates' işlemi bir liste üzerinde yapıldığı için her seferinde O(K) sürede çalışır. Bu da toplam karmaşıklığı daha da kötüleştirir.",
    "Pythonic Olmayan Yazım: Python'da indeksler üzerinden (range(len(lst))) dönmek yerine doğrudan elemanlar üzerinden dönmek veya 'set' veri yapısını kullanmak daha profesyonel bir yaklaşımdır.",
    "Gereksiz Karşılaştırmalar: Aynı eleman çiftleri defalarca kontrol edilmektedir. Bir 'seen' (görülenler) kümesi (set) kullanarak tek bir geçişte (O(N)) sonuç elde edilebilir."
  ],
  "puan": 3
}
```
[3/3] Reporter...


Merhaba, kıdemli Python geliştiricisi olarak her iki uzmanın raporlarını titizlikle inceledim ve birleştirdim. Mevcut kodun hem performans hem de Python standartları açısından ciddi iyileştirmelere ihtiyacı olduğu görülmektedir.

İşte birleştirilmiş final raporu:

### **Genel Değerlendirme**
İncelenen kod, işlevsel olarak doğru çalışsa da mimari açıdan oldukça verimsizdir. İç içe döngülerin kullanımı ve yanlış veri yapısı seçimi (liste üzerinden üyelik kontrolü), algoritmanın büyük veri setlerinde kilitlenmesine neden olacak düzeyde performans kaybı yaratmaktadır. Kod, modern Python yazım standartlarından (Pythonic) uzaktır ve ölçeklenebilirlik açısından risk taşımaktadır.

---

### **Tüm Sorunlar**

#### **1. Algoritmik Karmaşıklık ve Performans**
*   **O(n²) Zaman Karmaşıklığı:** İç içe geçmiş döngüler (nested loops) kullanımı nedeniyle veri miktarı arttıkça işlem süresi karesel olarak artmaktadır.
*   **Verimsiz Üyelik Kontrolü:** `duplicates` listesi üzerinde yapılan `not in` kontr

---
### Özet

Bu notebook'ta üç aşamalı bir geçiş yaptık:

| Aşama | Yetenek | Sınır |
|---|---|---|
| 1 — Basit Agent | Rollendirme, yapılandırılmış çıktı | Kodu test edemez |
| 2 — Tool Agent | Kodu çalıştırır, gözlemler, günceller | Tek odak noktası |
| 3 — Multi-Agent | Paralel uzmanlık, daha güvenilir çıktı | Koordinasyon maliyeti |

**Sonraki adım:** Bu yapıya state yönetimi eklendiğinde — agent'ların birbirinin çıktısını okuduğu paylaşımlı bir nesne — daha karmaşık ve bağımsız sistemler inşa etmek mümkün hale gelir. Bu noktada LangGraph gibi framework'ler devreye girer.